In [17]:
from dotenv import load_dotenv

load_dotenv()

True

# LangChain Runtime 运行时机制
LangChain的Runtime机制是理解Agent内部运行状态的关键。它包括三个核心概念：
| 概念 | 说明 | 生命周期 | 状态 |
|:-------|:--------:|-------:|-------:|
| Context  | 运行时上下文，传递配置参数      | 单次请求     | 只读 |
| state | 短期记忆，存储Agent当前对话信息、任务状态   | 同一会话，同一thread_id     | 可读写 |
| store | 长期记忆，包含用户偏好、失败经验等      | 跨会话     | 可读写  |

## 1.Context(运行时上下文)
Runtime Context是一种依赖注入机制：调用者在运行开始时把用户信息、租户信息、权限或外部服务连接传给Agent，
节点、中间件和工具在执行的时候再从Runtime读取。

它解决的是三个工程问题：
1. 避免把用户id、数据库连接等信息硬编码进工具
2. 同一个Agent可以安全地服务不同的用户和租户
3. 工具的业务参数只保留真正需要模型填写的内容

作用： 把隐私数据和Agent解耦

### 1.1 准备工作
准备一个字典，模拟一个多租户的在线教育项目数据库，其中的租户就是学校，user就是学员：
- 字典key是租户tenant_id，字典value是租户对应的user信息，也是字典
- user字典的key是user_id

In [18]:
USER_DATABASE = {
  'school_a': {
    'u001': {'name': '胡歌', 'course': 'LangChain 开发指南'},
    'u002': {'name': '杨幂', 'course': 'Java开发入门'}
  },
  'school_b': {
    'u001': {'name': '霍建华', 'course': 'Python 入门'},
    'u002': {'name': '刘诗诗', 'course': 'Langgraph+RAG实战'}
  }
}

### 1.2 定义ContextSchema

In [19]:
from dataclasses import dataclass

# dataclass声明类帮助省去写魔法函数了
# frozen表示只读，之后就不能动态再添加字段了，目前就这两个字段
@dataclass(frozen=True) 
class UserContext:
  '''Agent运行时上下文'''
  tenant_id: str
  user_id: str

### 1.3 在Tool中使用Context

In [20]:
from langchain.tools import tool, ToolRuntime

@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
  '''查询当前登录用户的信息'''
  
  ctx = runtime.context
  print('ctx:', ctx)

  # 根据租户id和用户id查询用户信息
  profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)
  return (
    f'姓名：{profile['name']}',
    f'课程：{profile['course']}'
  )

### 1.4 给Agent添加Context

In [21]:
from langchain.agents import create_agent

agent = create_agent(
  'deepseek-flash',
  context_schema=UserContext,  # 指定Context类型
  tools=[get_current_user_profile],
  system_prompt='你是课程平台助手，需要用户资料时调用工具。'
)

### 1.5 调用Agent，传递Context信息

In [22]:
from langchain.messages import HumanMessage

response = agent.invoke(
  {'messages': [HumanMessage('帮我查询正在学习的课程')]},
  context=UserContext(tenant_id='school_a', user_id='u001')
)

# print(response)
for msg in response['messages']:
  msg.pretty_print()

ctx: UserContext(tenant_id='school_a', user_id='u001')
================================ Human Message =================================

帮我查询正在学习的课程
================================== Ai Message ==================================

我来帮您查询。首先获取一下您的用户信息。
Tool Calls:
  get_current_user_profile (call_00_JZumJ0zwfEuBbHkEeYa59660)
 Call ID: call_00_JZumJ0zwfEuBbHkEeYa59660
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

["姓名：胡歌", "课程：LangChain 开发指南"]
================================== Ai Message ==================================

您好，**胡歌**！根据您的账号信息，您当前正在学习的课程是：

📚 **《LangChain 开发指南》**

需要我帮您查询这门课程的更多信息（如学习进度、章节内容等）吗？如果有其他需求，也请随时告诉我～


## 2. State (短期记忆)
State是Agent的短期记忆，存储当前会话的历史消息和任务状态等信息。

之前学习的短期记忆是AgentState，它只返回历史消息messages字段，但是可以基于它自定义State

### 2.1 自定义State

In [23]:
from langchain.agents import AgentState

class CustomState(AgentState):
  model_call_count: int

### 2.2 在工具中通过runtime访问state

In [24]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_model_call_count(runtime: ToolRuntime):
  ''' 一个更新state调用次数的工具 '''
  count = runtime.state.get('model_call_count', 0)
  print('调用次数：', count)

  update = {
    'model_call_count': count + 1,
    'messages': [
      ToolMessage(
        content='成功更新agent state的调用次数',
        tool_call_id=runtime.tool_call_id
      )
    ]
  }

  return Command(update=update)

### 2.3 创建Agent并指定state类型

In [25]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model('deepseek-flash', extra_body={'thinking': {'type': 'disabled'}})

agent = create_agent(
  model,
  tools=[update_model_call_count],
  state_schema=CustomState,
  checkpointer=InMemorySaver(),
  system_prompt='你是ai助手，你需要在每次回答用户前调用工具记录模型调用次数'
)

In [26]:
from langchain.messages import HumanMessage

config = {'configurable': {'thread_id': '1'}}

response = agent.invoke(
  {'messages': HumanMessage('为什么叫多肉植物啊？')},
  config
)

for msg in response['messages']:
  msg.pretty_print()

调用次数： 0
================================ Human Message =================================

为什么叫多肉植物啊？
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_8ErAC09fgLw7bb2ClyBF2049)
 Call ID: call_00_8ErAC09fgLw7bb2ClyBF2049
  Args:
================================= Tool Message =================================
Name: update_model_call_count

成功更新agent state的调用次数
================================== Ai Message ==================================

“多肉植物”这个名字，主要是从它们的**外形特征**来的：

1. **“多肉”指的是肉质化的器官**
多肉植物的根、茎、叶中至少有一部分变得肥厚、多汁，看起来“肉嘟嘟”的。这些肥厚的部分里储存着大量水分，所以叫“多肉”。比如叶片厚厚的玉露、熊童子，或者茎干粗壮的仙人掌类。

2. **本质是储水适应**
它们多原产于干旱或季节性干旱地区，把水存在体内，等到缺水时再慢慢用，所以肉质化是一种“抗旱装备”。

3. **不是植物学分类，而是俗称**
“多肉植物”不是严格的分类学名词，而是园艺上的统称，包含景天科、仙人掌科、番杏科、百合科（部分）等多个科里具有肉质储水特征的植物。所以只要“肉厚、能储水”，很多不同家族的植物都会被叫作多肉。

简单说：**因为它们的叶或茎肥厚多汁、像“肉”一样储水，所以叫多肉植物。**


In [27]:
# 可以查询state状态，传递相同的thread_id
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='为什么叫多肉植物啊？', additional_kwargs={}, response_metadata={}, id='0e965d7b-b988-496b-9776-3cea9edb5428'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 275, 'total_tokens': 298, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 147}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'b070b1e1-83a5-457d-a0c6-ef5b46243683', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1fcd6-2e64-768a-892b-a7a0431ba08a-0', tool_calls=[{'name': 'update_model_call_count', 'args': {}, 'id': 'call_00_8ErAC09fgLw7bb2ClyBF2049', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

In [28]:
response = agent.invoke(
  {'messages': HumanMessage('你知道小狗钱钱吗？')},
  config
)

for msg in response['messages']:
  msg.pretty_print()

调用次数： 1
================================ Human Message =================================

为什么叫多肉植物啊？
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_8ErAC09fgLw7bb2ClyBF2049)
 Call ID: call_00_8ErAC09fgLw7bb2ClyBF2049
  Args:
================================= Tool Message =================================
Name: update_model_call_count

成功更新agent state的调用次数
================================== Ai Message ==================================

“多肉植物”这个名字，主要是从它们的**外形特征**来的：

1. **“多肉”指的是肉质化的器官**
多肉植物的根、茎、叶中至少有一部分变得肥厚、多汁，看起来“肉嘟嘟”的。这些肥厚的部分里储存着大量水分，所以叫“多肉”。比如叶片厚厚的玉露、熊童子，或者茎干粗壮的仙人掌类。

2. **本质是储水适应**
它们多原产于干旱或季节性干旱地区，把水存在体内，等到缺水时再慢慢用，所以肉质化是一种“抗旱装备”。

3. **不是植物学分类，而是俗称**
“多肉植物”不是严格的分类学名词，而是园艺上的统称，包含景天科、仙人掌科、番杏科、百合科（部分）等多个科里具有肉质储水特征的植物。所以只要“肉厚、能储水”，很多不同家族的植物都会被叫作多肉。

简单说：**因为它们的叶或茎肥厚多汁、像“肉”一样储水，所以叫多肉植物。**
================================ Human Message =================================

你知道小狗钱钱吗？
==========================

In [29]:
response = agent.invoke(
  {'messages': [HumanMessage('让我想到了9月份在看的一本书《小岛经济学》')]},
  config
)

for msg in response['messages']:
  msg.pretty_print()

调用次数： 2
================================ Human Message =================================

为什么叫多肉植物啊？
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_8ErAC09fgLw7bb2ClyBF2049)
 Call ID: call_00_8ErAC09fgLw7bb2ClyBF2049
  Args:
================================= Tool Message =================================
Name: update_model_call_count

成功更新agent state的调用次数
================================== Ai Message ==================================

“多肉植物”这个名字，主要是从它们的**外形特征**来的：

1. **“多肉”指的是肉质化的器官**
多肉植物的根、茎、叶中至少有一部分变得肥厚、多汁，看起来“肉嘟嘟”的。这些肥厚的部分里储存着大量水分，所以叫“多肉”。比如叶片厚厚的玉露、熊童子，或者茎干粗壮的仙人掌类。

2. **本质是储水适应**
它们多原产于干旱或季节性干旱地区，把水存在体内，等到缺水时再慢慢用，所以肉质化是一种“抗旱装备”。

3. **不是植物学分类，而是俗称**
“多肉植物”不是严格的分类学名词，而是园艺上的统称，包含景天科、仙人掌科、番杏科、百合科（部分）等多个科里具有肉质储水特征的植物。所以只要“肉厚、能储水”，很多不同家族的植物都会被叫作多肉。

简单说：**因为它们的叶或茎肥厚多汁、像“肉”一样储水，所以叫多肉植物。**
================================ Human Message =================================

你知道小狗钱钱吗？
==========================

In [30]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='为什么叫多肉植物啊？', additional_kwargs={}, response_metadata={}, id='0e965d7b-b988-496b-9776-3cea9edb5428'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 275, 'total_tokens': 298, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 147}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'b070b1e1-83a5-457d-a0c6-ef5b46243683', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1fcd6-2e64-768a-892b-a7a0431ba08a-0', tool_calls=[{'name': 'update_model_call_count', 'args': {}, 'id': 'call_00_8ErAC09fgLw7bb2ClyBF2049', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

## 3. Store (长期记忆-跨会话使用)

### 3.1 初始化Store

In [31]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

### 3.2 定义namespace和key

In [32]:
def perference_namespace(ctx: UserContext) -> tuple[str, ...]:
  return (
    'heima_platform',
    ctx.tenant_id,
    ctx.user_id
  )

key = 'learning_perferences'

### 3.3 定义工具，访问和修改Store

In [33]:
from langchain.tools import tool, ToolRuntime

@tool
def save_learning_perference(
  p_name: str,
  p_value: str,
  runtime: ToolRuntime[UserContext]
) -> str:
  ''' 保存当前用户的学习偏好 '''
  context = runtime.context
  # 通过ctx来构建namespace
  namespace = perference_namespace(context)
  # 调用get方法获取namespace和key对应的item
  item = runtime.store.get(namespace, key)
  perferences = item.value if item else {}
  # 记录本次要保存的用户偏好
  perferences[p_name] = p_value
  # 保存在store里
  runtime.store.put(namespace, key, perferences)

  return f'已保存好偏好：{p_name}={p_value}'

@tool
def get_learning_perference(runtime: ToolRuntime[UserContext]) -> str:
  ''' 获取用户的学习偏好工具 '''
  namespace = perference_namespace(runtime.context)
  item = runtime.store.get(namespace, key)
  if not item:
    return '当前用户还没有学习偏好设定'
  return f'当前学习偏好为{item.value}'


### 3.4 创建Agent

In [34]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

checkpointer = InMemorySaver()

agent = create_agent(
  'deepseek-flash',
  tools=[save_learning_perference, get_learning_perference, get_current_user_profile],
  context_schema=UserContext,
  checkpointer=checkpointer,
  store=store,
  system_prompt=(
    '你是课程平台助手'
    '涉及用户资料时调用资料工具'
    '涉及长期学习偏好或给用户讲解代码时调用偏好工具'
  )
)

In [38]:
context = UserContext(tenant_id='school_a', user_id='u001')

response = agent.invoke(
  {'messages': [HumanMessage('记住：我喜欢先看代码，再看原理')]},
  config={'configurable': {'thread_id':'thread_01'}},
  context=context
)

for msg in response['messages']:
  msg.pretty_print()

================================ Human Message =================================

记住：我喜欢先看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_perference (call_00_SmT2sSggWPKmWrJS4u9e5058)
 Call ID: call_00_SmT2sSggWPKmWrJS4u9e5058
  Args:
    p_name: 学习顺序偏好
    p_value: 先看代码，再看原理
================================= Tool Message =================================
Name: save_learning_perference

已保存好偏好：学习顺序偏好=先看代码，再看原理
================================== Ai Message ==================================

已经记住了 ✅

**学习顺序偏好：先看代码，再看原理**

之后我在讲解时，会按这个顺序来：
1. 先给出可运行的代码示例（带关键注释）
2. 再解释背后的原理，以及为什么要这样实现
3. 最后补充边界情况、常见坑点和延伸阅读

如果你有具体想学的主题，把代码发给我或者告诉我方向，我就按这个节奏来。


In [39]:
result = agent.invoke(
  {'messages': [HumanMessage('LangChain的结构化输出是怎么做的？')]},
  config={'configurable': {'thread_id': 'thread_02'}},
  context=context
)

for msg in result['messages']:
  msg.pretty_print()

================================ Human Message =================================

LangChain的结构化输出是怎么做的？
================================== Ai Message ==================================
Tool Calls:
  get_learning_perference (call_00_WgmpURNETlygT9o31QDz2088)
 Call ID: call_00_WgmpURNETlygT9o31QDz2088
  Args:
================================= Tool Message =================================
Name: get_learning_perference

当前学习偏好为{'学习顺序偏好': '先看代码，再看原理'}
================================== Ai Message ==================================

按你的偏好，先上代码，再讲原理。

## 一、代码：三种常用做法

### 1. `with_structured_output`（最主流，Pydantic 版）

```python
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class Person(BaseModel):
    """一个人的基本信息。"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄，整数")
    skills: list[str] = Field(default_factory=list, description="技能列表")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

structured_llm = llm.with_structured_out

In [40]:
response = agent.invoke(
  {'messages': [HumanMessage('我正在学习Langchain，Langchain的作用用简单的话语说一下')]},
  config={'configurable': {'thread_id':'thread_03'}},
  context=context
)

for msg in response['messages']:
  msg.pretty_print()

ctx: UserContext(tenant_id='school_a', user_id='u001')
================================ Human Message =================================

我正在学习Langchain，Langchain的作用用简单的话语说一下
================================== Ai Message ==================================
Tool Calls:
  get_learning_perference (call_00_qRs1Edvn4aqyTUmidCyo0279)
 Call ID: call_00_qRs1Edvn4aqyTUmidCyo0279
  Args:
  get_current_user_profile (call_01_DUlpIreG2k8joqbb2Eec1721)
 Call ID: call_01_DUlpIreG2k8joqbb2Eec1721
  Args:
================================= Tool Message =================================
Name: get_learning_perference

当前学习偏好为{'学习顺序偏好': '先看代码，再看原理'}
================================= Tool Message =================================
Name: get_current_user_profile

["姓名：胡歌", "课程：LangChain 开发指南"]
================================== Ai Message ==================================

胡歌你好！我记得你的学习习惯是**先看代码，再看原理**，那我们先上一小段代码：这是用 LangChain 实现“问一句话，再根据这句话继续追问”的最简例子：

```python
from langchain_openai import ChatOpenAI
from lan